In [4]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

name, memory.total [MiB], driver_version
Tesla T4, 15360 MiB, 580.82.07


In [5]:
import platform

import torch
import torchvision

print(platform.python_version(), torch.__version__, torchvision.__version__, torch.version.cuda)
assert torch.cuda.is_available()

3.13.16 2.14.1+cu130 0.29.1+cu130 13.0


In [6]:
from google.colab import drive, userdata

try:
    repo_url = f"https://{userdata.get('GH_TOKEN')}@github.com/duyenphung206-sudo/vit-oxford-pets.git"
except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
    repo_url = "https://github.com/duyenphung206-sudo/vit-oxford-pets.git"
!git clone -q {repo_url} /content/vit-oxford-pets
%cd /content/vit-oxford-pets
!pip install -q -r requirements.txt
drive.mount("/content/drive")

fatal: destination path '/content/vit-oxford-pets' already exists and is not an empty directory.
/content/vit-oxford-pets
Mounted at /content/drive


In [7]:
from torchvision import models

weights = models.ViT_B_16_Weights.IMAGENET1K_V1
vit = models.vit_b_16(weights=weights)
n_params = sum(p.numel() for p in vit.parameters())
print(f"{n_params:,}", weights.meta["_metrics"]["ImageNet-1K"])
print(weights.transforms())
assert n_params == 86_567_656

Downloading: "https://download.pytorch.org/models/vit_b_16-c867db91.pth" to /root/.cache/torch/hub/checkpoints/vit_b_16-c867db91.pth


100%|██████████| 330M/330M [00:04<00:00, 80.8MB/s]


86,567,656 {'acc@1': 81.072, 'acc@5': 95.318}
ImageClassification(
    crop_size=[224]
    resize_size=[256]
    mean=[0.485, 0.456, 0.406]
    std=[0.229, 0.224, 0.225]
    interpolation=InterpolationMode.BILINEAR
)


In [8]:
from src.models.factory import build_model
from src.utils import count_parameters

for strategy, k in [("linear_probe", 0), ("partial", 4), ("full", 0)]:
    cfg = {"name": "vit_b_16", "num_classes": 37, "strategy": strategy, "unfreeze_last_k": k}
    p = count_parameters(build_model(cfg))
    print(f"{strategy:<13} {p['trainable']:>11,} / {p['total']:,}")

linear_probe       28,453 / 85,827,109
partial        28,381,477 / 85,827,109
full           85,827,109 / 85,827,109


In [9]:
import time

device = torch.device("cuda")
model = build_model({"name": "vit_b_16", "num_classes": 37, "strategy": "full"}).to(device)
optimizer = torch.optim.SGD(model.parameters(), lr=0.003, momentum=0.9)
scaler = torch.amp.GradScaler("cuda")
loss_fn = torch.nn.CrossEntropyLoss()

for batch_size in (32, 64):
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    images = torch.randn(batch_size, 3, 224, 224, device=device)
    labels = torch.randint(0, 37, (batch_size,), device=device)
    timings = []
    for _ in range(6):
        torch.cuda.synchronize()
        start = time.perf_counter()
        optimizer.zero_grad(set_to_none=True)
        with torch.autocast(device_type="cuda"):
            loss = loss_fn(model(images), labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        torch.cuda.synchronize()
        timings.append(time.perf_counter() - start)
    step_ms = 1000 * sum(timings[1:]) / len(timings[1:])
    print(f"batch {batch_size:>3} | {step_ms:6.0f} ms/step | {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB")

batch  32 |    270 ms/step | 3.0 GiB
batch  64 |    521 ms/step | 5.2 GiB


In [10]:
!python -m src.train --config configs/e4_vit_full_ft.yaml --fake-data --epochs 1 --output-dir /content/smoke --checkpoint-dir /content/smoke_ckpt
!cat /content/smoke/metrics.json

e4_vit_full_ft | seed 42 | cuda | trainable 85,827,109 / 85,827,109
epoch   1/1 | train 3.6118 / 0.0234 | val 3.6106 / 0.0391 | 6s
best epoch 1 | val loss 3.6106 | val acc 0.0391
{
  "experiment": "e4_vit_full_ft",
  "seed": 42,
  "best_epoch": 1,
  "best_val_loss": 3.610559,
  "best_val_acc": 0.039062,
  "epochs_run": 1,
  "parameters_total": 85827109,
  "parameters_trainable": 85827109,
  "mean_epoch_time_s": 5.85,
  "peak_gpu_memory_mb": 5291.8,
  "device": "Tesla T4",
  "torch": "2.14.1+cu130",
  "torchvision": "0.29.1+cu130",
  "python": "3.13.16",
  "checkpoint": "/content/smoke_ckpt/best.pt",
  "fake_data": true
}

In [11]:
!pip freeze | grep -i -E "^(torch|torchvision|numpy|pandas|matplotlib|scikit-learn|pyyaml|tqdm|pillow|gradio|pytest)=="

gradio==6.31.0
matplotlib==3.10.0
numpy==2.4.6
pandas==2.2.3
pillow==11.3.0
pytest==8.4.2
PyYAML==6.0.3
scikit-learn==1.6.1
torch==2.14.1+cu130
torchvision==0.29.1+cu130
tqdm==4.67.3
